# Notebook 02 - Sparse Autoencoders

Notebook 01 showed that information about the output language appears in the model's early residual-stream layers. We represented this information with one dense steering direction,

$$
v^{\mathrm{it-en}}_{\ell} = \mu^{\mathrm{it}}_{\ell} - \mu^{\mathrm{en}}_{\ell},
$$

and showed that adding or subtracting this direction can change the language used during generation.

This is already a useful internal intervention, but a residual-stream vector is still difficult to interpret. Each activation has thousands of coordinates, and neural networks can represent more features than they have dimensions by storing several features in overlapping directions. This phenomenon is called **superposition**. Consequently, a single residual-stream direction may mix language with other properties of the text.

Sparse Autoencoders (SAEs) try to recover a more interpretable feature basis. Instead of treating the residual-stream coordinates themselves as features, an SAE learns a large dictionary of feature directions and represents each residual vector using only a small number of active dictionary elements.

In this notebook we will:

1. load a TopK SAE trained on the layer-4 residual stream of `Qwen/Qwen3-1.7B-Base`;
2. encode English and Italian residual vectors into sparse feature activations;
3. identify features that activate differently for the two languages;
4. add or remove Italian-associated decoder directions during generation;
5. inspect whether those features generalize to unseen text.


<p align="center">
  <img src="https://raw.githubusercontent.com/leobertolazzi/mech-interp-lab-lcl26/main/images/sae.png" alt="Single-layer sparse autoencoder" width="450"/>
</p>

*Sparse autoencoder: although the hidden layer can contain many possible features, only a small subset is active for a given input.*


## 0. Colab Setup

This cell installs the small package set needed in Colab. Use a GPU runtime: the notebook loads one language model and one layer-4 SAE checkpoint.


In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    packages = [
        "transformers>=5.10.1",
        "accelerate>=0.33.0",
        "huggingface_hub>=1.17.0",
        "pandas>=2.0.0",
        "matplotlib>=3.8.0",
        "tqdm>=4.66.0",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
import re
import json
from pathlib import Path
import requests
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_DIR = Path("data")

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
SAE_REPO = "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_100"
SAE_LAYERS = [4]
PRIMARY_SAE_LAYER = 4
TOP_K = 100
BATCH_SIZE = 4
MAX_SENTENCES_PER_LANGUAGE = 50
MAX_NEW_TOKENS = 120

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

LANGUAGE_COLORS = {
    "english": "#4C72B0",
    "italian": "#55A868",
    "unknown": "#A0A0A0",
}
LANGUAGE_ORDER = ["english", "italian", "unknown"]
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
})

print(f"Device: {DEVICE}")
print(f"SAE layer: {PRIMARY_SAE_LAYER}")


## 1. Load Data and Model

As a first step, we will reuse some of the code from Notebook 01 for the following purposes:
1. Reading the JSON data of English and Italian sentences and questions.
2. Loading the base Qwen model
3. For each English/Italian sentence, we run a normal forward pass and save the residual-stream vector from layer 4.


In [ ]:
languages = [
    "italian",
    "english",
]

file_types = [
    "sentences",
    "test_questions",
    "unseen_sentences",
]

base_url = "https://raw.githubusercontent.com/leobertolazzi/mech-interp-lab-lcl26/main/data"

DATA_DIR.mkdir(exist_ok=True)

for language in languages:
    for file_type in file_types:
        filename = f"{language}_{file_type}.json"
        url = f"{base_url}/{filename}"

        response = requests.get(url)
        response.raise_for_status()

        (DATA_DIR / filename).write_bytes(response.content)

print("All files downloaded.")

In [ ]:
def read_json_list(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. In Colab, clone/download the repo or upload the data/ directory first."
        )
    data = json.loads(path.read_text())
    return data

english_sentences = read_json_list(DATA_DIR / "english_sentences.json")[:MAX_SENTENCES_PER_LANGUAGE]
italian_sentences = read_json_list(DATA_DIR / "italian_sentences.json")[:MAX_SENTENCES_PER_LANGUAGE]
english_test_questions = read_json_list(DATA_DIR / "english_test_questions.json")
italian_test_questions = read_json_list(DATA_DIR / "italian_test_questions.json")
english_unseen_sentences = read_json_list(DATA_DIR / "english_unseen_sentences.json")
italian_unseen_sentences = read_json_list(DATA_DIR / "italian_unseen_sentences.json")

print(f"English sentences: {len(english_sentences)}")
print(f"Italian sentences: {len(italian_sentences)}")
print(f"English test questions: {len(english_test_questions)}")
print(f"Italian test questions: {len(italian_test_questions)}")
print(f"Unseen English sentences: {len(english_unseen_sentences)}")
print(f"Unseen Italian sentences: {len(italian_unseen_sentences)}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print(f"Model device: {MODEL_DEVICE}")


In [ ]:
def _layer_output_to_hidden_states(output: Any) -> torch.Tensor:
    """Return the hidden-state tensor when a transformer layer returns either a tensor or a tuple."""
    return output[0] if isinstance(output, tuple) else output


def capture_residual_batch(texts: list[str], layers: list[int] = SAE_LAYERS) -> dict[int, torch.Tensor]:
    captured_hidden_states: dict[int, torch.Tensor] = {}
    hook_handles = []

    def make_capture_hook(layer: int):
        def hook_fn(module, inputs, output):
            hidden_states = _layer_output_to_hidden_states(output)
            captured_hidden_states[layer] = hidden_states.detach()
        return hook_fn

    for layer in layers:
        layer_module = model.model.layers[layer]
        hook_handles.append(layer_module.register_forward_hook(make_capture_hook(layer)))


    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    encoded = {name: value.to(MODEL_DEVICE) for name, value in encoded.items()}
    with torch.no_grad():
        model(**encoded, use_cache=False)

    for handle in hook_handles:
        handle.remove()

    last_token_positions = encoded["attention_mask"].sum(dim=1) - 1
    batch_positions = torch.arange(len(texts), device=MODEL_DEVICE)
    residuals_by_layer = {
        layer: hidden_states[batch_positions, last_token_positions].float().cpu()
        for layer, hidden_states in captured_hidden_states.items()
    }
    return residuals_by_layer


def extract_residuals_by_layer(
    texts: list[str],
    layers: list[int] = SAE_LAYERS,
    batch_size: int = BATCH_SIZE,
) -> dict[int, torch.Tensor]:
    residual_batches_by_layer = {layer: [] for layer in layers}
    for start in tqdm(range(0, len(texts), batch_size), desc="Extract residuals"):
        batch_texts = texts[start:start + batch_size]
        batch_residuals = capture_residual_batch(batch_texts, layers=layers)
        for layer in layers:
            residual_batches_by_layer[layer].append(batch_residuals[layer])
    return {
        layer: torch.cat(residual_batches, dim=0)
        for layer, residual_batches in residual_batches_by_layer.items()
    }


def compute_residuals(
    language: str,
    texts: list[str],
    layers: list[int] = SAE_LAYERS,
) -> dict[int, torch.Tensor]:
    residuals_by_layer = {}

    print(f"Computing {language} residuals for layers: {layers}")
    computed_residuals = extract_residuals_by_layer(texts, layers=layers)
    for layer, residuals in computed_residuals.items():
        residuals_by_layer[layer] = residuals

    return residuals_by_layer

english_residuals_by_layer = compute_residuals("english", english_sentences, layers=SAE_LAYERS)
italian_residuals_by_layer = compute_residuals("italian", italian_sentences, layers=SAE_LAYERS)

for layer in SAE_LAYERS:
    print(
        f"Layer {layer:02d}: "
        f"english={tuple(english_residuals_by_layer[layer].shape)} "
        f"italian={tuple(italian_residuals_by_layer[layer].shape)}"
    )


## 2. Load the Layer-4 Qwen-Scope SAE

We now replace the original residual-stream coordinate system with a learned sparse feature dictionary. Qwen-Scope provides one residual-stream SAE checkpoint for each model layer; here we load only `layer4.sae.pt`.

Let

$$
h \in \mathbb{R}^{d_{\text{model}}}
$$

be a residual-stream vector. An SAE first maps it into a much wider feature space:

$$
u = W_{\text{enc}}h + b_{\text{enc}},
$$

where $u \in \mathbb{R}^{d_{\text{SAE}}}$ contains the feature pre-activations and usually $d_{\text{SAE}} \gg d_{\text{model}}$.

A **TopK SAE** keeps only the $K$ largest feature activations:

$$
z_j =
\begin{cases}
u_j, & j \in \operatorname{TopK}(u),\\
0, & \text{otherwise}.
\end{cases}
$$

Thus $z$ is sparse: at most $K$ of its $d_{\text{SAE}}$ dimensions are non-zero for each residual vector. In this checkpoint, $K=100$.

The decoder reconstructs the original residual vector:

$$
\hat{h} = W_{\text{dec}}z + b_{\text{dec}}.
$$

The SAE is trained so that $\hat{h}$ remains close to $h$ while the latent representation stays sparse. For a TopK SAE, the sparsity constraint is enforced directly by the TopK operation rather than only through a soft sparsity penalty.


In [ ]:
def load_qwen_scope_sae(layer: int) -> dict[str, torch.Tensor]:
    sae_path = hf_hub_download(repo_id=SAE_REPO, filename=f"layer{layer}.sae.pt")
    sae_state = torch.load(sae_path, map_location="cpu")
    sae = {
        "W_enc": sae_state["W_enc"].to(DEVICE).float(),  # [d_sae, d_model]
        "W_dec": sae_state["W_dec"].to(DEVICE).float(),  # [d_model, d_sae]
        "b_enc": sae_state["b_enc"].to(DEVICE).float(),
        "b_dec": sae_state["b_dec"].to(DEVICE).float(),
    }
    d_sae, d_model = sae["W_enc"].shape
    print(f"Layer {layer:02d}: SAE width={d_sae}, d_model={d_model}")
    return sae

sae_by_layer = {layer: load_qwen_scope_sae(layer) for layer in SAE_LAYERS}


In [ ]:
def encode_topk_sae(
    residuals: torch.Tensor,
    sae: dict[str, torch.Tensor],
    batch_size: int = 16,
) -> torch.Tensor:
    sparse_activation_batches = []
    residuals = residuals.float()
    W_enc = sae["W_enc"]
    b_enc = sae["b_enc"]

    for start in tqdm(range(0, residuals.shape[0], batch_size), desc="Encode SAE"):
        residual_batch = residuals[start:start + batch_size].to(DEVICE)
        feature_pre_activations = residual_batch @ W_enc.T + b_enc
        top_values, top_indices = feature_pre_activations.topk(TOP_K, dim=-1)

        sparse_activations = torch.zeros_like(feature_pre_activations)
        sparse_activations.scatter_(dim=-1, index=top_indices, src=top_values)
        sparse_activation_batches.append(sparse_activations.cpu())

    return torch.cat(sparse_activation_batches, dim=0)


sae_activations_by_layer = {}
for layer in SAE_LAYERS:
    english_sae_activations = encode_topk_sae(
        english_residuals_by_layer[layer], sae_by_layer[layer]
    )
    italian_sae_activations = encode_topk_sae(
        italian_residuals_by_layer[layer], sae_by_layer[layer]
    )
    print(f"Layer {layer:02d}: computed SAE activations.")

    sae_activations_by_layer[layer] = {
        "english": english_sae_activations,
        "italian": italian_sae_activations,
    }
    print(
        f"  english={tuple(english_sae_activations.shape)} "
        f"italian={tuple(italian_sae_activations.shape)}"
    )


## 3. Differential Feature Discovery at Layer 4

The SAE converts every residual vector into a sparse activation vector $z$. We can therefore repeat the contrastive analysis from Notebook 01, but now one coordinate corresponds to one learned SAE feature rather than one original residual-stream dimension.

For each feature $j$, we compute

$$
\Delta_j =
\frac{1}{N_{\text{it}}}\sum_{i=1}^{N_{\text{it}}} z_j(x_i^{\text{it}})
-
\frac{1}{N_{\text{en}}}\sum_{i=1}^{N_{\text{en}}} z_j(x_i^{\text{en}}).
$$

Positive $\Delta_j$ means that feature $j$ is more active on Italian sentences; negative $\Delta_j$ means it is more active on English sentences.

Unlike the dense steering vector, which summarized the entire contrast with one direction, this analysis lets us inspect individual sparse features.


In [ ]:
def qwen_scope_hf_url(layer: int) -> str:
    return f"https://huggingface.co/{SAE_REPO}/blob/main/layer{layer}.sae.pt"


def differential_features(
    english_activations: torch.Tensor,
    italian_activations: torch.Tensor,
    layer: int,
    top_n: int = 10,
) -> pd.DataFrame:
    english_mean_activation = english_activations.mean(dim=0)
    italian_mean_activation = italian_activations.mean(dim=0)
    italian_minus_english = italian_mean_activation - english_mean_activation

    top_italian = torch.topk(italian_minus_english, top_n)
    top_english = torch.topk(-italian_minus_english, top_n)

    rows = []
    for rank, (feature_id, score) in enumerate(
        zip(top_italian.indices.tolist(), top_italian.values.tolist()), start=1
    ):
        rows.append({
            "layer": layer,
            "associated_language": "italian",
            "rank": rank,
            "feature_id": feature_id,
            "italian_minus_english": score,
            "english_mean_activation": english_mean_activation[feature_id].item(),
            "italian_mean_activation": italian_mean_activation[feature_id].item(),
            "qwen_scope_checkpoint": qwen_scope_hf_url(layer),
        })
    for rank, (feature_id, score) in enumerate(
        zip(top_english.indices.tolist(), top_english.values.tolist()), start=1
    ):
        rows.append({
            "layer": layer,
            "associated_language": "english",
            "rank": rank,
            "feature_id": feature_id,
            "italian_minus_english": -score,
            "english_mean_activation": english_mean_activation[feature_id].item(),
            "italian_mean_activation": italian_mean_activation[feature_id].item(),
            "qwen_scope_checkpoint": qwen_scope_hf_url(layer),
        })
    return pd.DataFrame(rows)

primary_layer_activations = sae_activations_by_layer[PRIMARY_SAE_LAYER]
feature_table = differential_features(
    primary_layer_activations["english"],
    primary_layer_activations["italian"],
    layer=PRIMARY_SAE_LAYER,
    top_n=10,
)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = feature_table.copy().sort_values("italian_minus_english")
colors = plot_data["associated_language"].map(LANGUAGE_COLORS)
feature_labels = [
    f"{row.associated_language[:2].upper()} F{row.feature_id}"
    for row in plot_data.itertuples()
]

ax.barh(feature_labels, plot_data["italian_minus_english"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Mean Italian activation - mean English activation")
ax.set_title(f"Layer {PRIMARY_SAE_LAYER}: top differential SAE features")
plt.tight_layout()
plt.show()


After identifying which features are on average more active on English or Italian sentences, we examine the full distribution of their activations in the two languages.

In [ ]:
def plot_feature_distributions(layer: int, feature_ids: list[int], title: str):
    english_activations = sae_activations_by_layer[layer]["english"]
    italian_activations = sae_activations_by_layer[layer]["italian"]
    fig, axes = plt.subplots(1, len(feature_ids), figsize=(3 * len(feature_ids), 3.1), sharey=False)
    if len(feature_ids) == 1:
        axes = [axes]
    for ax, feature_id in zip(axes, feature_ids):
        ax.hist(
            english_activations[:, feature_id].numpy(),
            alpha=0.72,
            label="english",
            bins=16,
            color=LANGUAGE_COLORS["english"],
        )
        ax.hist(
            italian_activations[:, feature_id].numpy(),
            alpha=0.72,
            label="italian",
            bins=16,
            color=LANGUAGE_COLORS["italian"],
        )
        ax.set_title(f"L{layer} F{feature_id}", fontsize=10)
        ax.set_xlabel("Feature activation")
    axes[0].set_ylabel("Sentence count")
    axes[0].legend(frameon=False)
    fig.suptitle(title, y=1.05, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def feature_selectivity_summary(
    layer: int,
    feature_ids: list[int],
    associated_language: str,
) -> pd.DataFrame:
    english_activations = sae_activations_by_layer[layer]["english"]
    italian_activations = sae_activations_by_layer[layer]["italian"]
    rows = []
    for feature_id in feature_ids:
        english_values = english_activations[:, feature_id].float()
        italian_values = italian_activations[:, feature_id].float()
        english_mean_activation = english_values.mean().item()
        italian_mean_activation = italian_values.mean().item()
        english_active_fraction = (english_values > 0).float().mean().item()
        italian_active_fraction = (italian_values > 0).float().mean().item()
        rows.append({
            "layer": layer,
            "associated_language": associated_language,
            "feature_id": feature_id,
            "english_mean_activation": english_mean_activation,
            "italian_mean_activation": italian_mean_activation,
            "english_active_fraction": english_active_fraction,
            "italian_active_fraction": italian_active_fraction,
            "target_minus_other_active_fraction": (
                english_active_fraction - italian_active_fraction
                if associated_language == "english"
                else italian_active_fraction - english_active_fraction
            ),
        })
    return pd.DataFrame(rows)

italian_feature_ids = (
    feature_table.query("associated_language == 'italian'")["feature_id"].head(5).tolist()
)
english_feature_ids = (
    feature_table.query("associated_language == 'english'")["feature_id"].head(5).tolist()
)

plot_feature_distributions(
    PRIMARY_SAE_LAYER,
    italian_feature_ids,
    f"Top Italian-associated SAE features at layer {PRIMARY_SAE_LAYER}",
)
plot_feature_distributions(
    PRIMARY_SAE_LAYER,
    english_feature_ids,
    f"Top English-associated SAE features at layer {PRIMARY_SAE_LAYER}",
)

selectivity_table = pd.concat([
    feature_selectivity_summary(PRIMARY_SAE_LAYER, italian_feature_ids, "italian"),
    feature_selectivity_summary(PRIMARY_SAE_LAYER, english_feature_ids, "english"),
], ignore_index=True)

## 4. Italian-Feature Intervention

The decoder gives each SAE feature an explicit direction in the model's residual-stream space. Let $d_j \in \mathbb{R}^{d_{\text{model}}}$ be column $j$ of $W_{\text{dec}}$. The SAE reconstruction can be written as

$$
\hat{h} = b_{\text{dec}} + \sum_{j=1}^{d_{\text{SAE}}} z_j d_j.
$$

This equation tells us how an SAE represents a residual vector: the residual is decomposed as the sum of the decoder directions of active features scaled by their activation plus a bias term.

This allow us to perform a very similar intervention to the one we saw in Notebook 01. For the set of Italian-associated features $I$, we define the fixed Italian direction

$$
v_{\text{it}} = \sum_{j \in I} d_j.
$$

The scalar $\alpha$ controls both the strength and the sign of the intervention:

$$
h'_{\ell,t} = h_{\ell,t} + \alpha v_{\text{it}}.
$$

Positive $\alpha$ adds the Italian feature direction and should encourage Italian.

Negative $\alpha$ subtracts the same direction and should suppress Italian.

The SAE encoder is not used for this intervention. A PyTorch forward hook adds the signed vector at the layer-4 residual stream during generation and is removed immediately afterward.


<p align="center">
  <img src="https://raw.githubusercontent.com/leobertolazzi/mech-interp-lab-lcl26/main/images/sae_steering.png" alt="Single-layer sparse autoencoder" width="450"/>
</p>

In [ ]:
ENGLISH_MARKERS = {
    "the", "and", "that", "this", "these", "those", "with", "from", "for", "are", "is", "was", "were",
    "be", "been", "being", "to", "of", "in", "on", "at", "by", "as", "it", "its", "they", "them", "their",
    "we", "you", "he", "she", "or", "if", "then", "than", "but", "not", "also", "when", "while", "because",
    "about", "between", "through", "into", "over", "under", "more", "less", "very", "can", "could", "should",
    "would", "will", "may", "might", "must", "have", "has", "had", "do", "does", "did",
}

ITALIAN_MARKERS = {
    "il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "di", "a", "da", "in", "con", "su", "per",
    "tra", "fra", "che", "e", "o", "ma", "se", "poi", "come", "quando", "mentre", "perché", "non", "anche",
    "più", "meno", "molto", "questo", "questa", "questi", "queste", "quello", "quella", "quelli", "quelle",
    "mi", "ti", "si", "ci", "vi", "lui", "lei", "loro", "noi", "voi", "suo", "sua", "è", "sono",
    "era", "erano", "essere", "stato", "stata", "hanno", "ha", "ho", "hai", "abbiamo", "può", "possono",
    "deve", "devono", "dovrebbe", "dovrebbero", "del", "dello", "della", "dei", "degli", "delle", "al", "allo",
    "alla", "ai", "agli", "alle", "nel", "nello", "nella", "nei", "negli", "nelle",
}


def classify_language_by_markers(text: str) -> str:
    tokens = re.findall(r"[a-zàèéìòù']+", text.lower())
    english_score = sum(token in ENGLISH_MARKERS for token in tokens)
    italian_score = sum(token in ITALIAN_MARKERS for token in tokens)

    if italian_score > english_score:
        return "italian"
    elif english_score > italian_score:
        return "english"
    else:
        return "unknown"


In [ ]:
PROMPT_STYLES = {
    "english": "Answer the question in English.",
    "italian": "Rispondi alla domanda in Italiano.",
}


def build_prompt(question: str, prompt_style: str) -> str:
    if prompt_style == "italian":
        return f"{PROMPT_STYLES[prompt_style]}\n\nDomanda: {question}\nRisposta:"
    return f"{PROMPT_STYLES[prompt_style]}\n\nQuestion: {question}\nAnswer:"


def make_italian_feature_hook(
    layer: int,
    feature_ids: list[int],
    alpha: float,
):
    sae = sae_by_layer[layer]
    W_dec = sae["W_dec"].to(MODEL_DEVICE)

    feature_ids = list(feature_ids)
    feature_tensor = torch.tensor(feature_ids, dtype=torch.long, device=MODEL_DEVICE)
    decoder_directions = W_dec[:, feature_tensor].float()
    italian_direction = decoder_directions.sum(dim=1)
    steering_direction = italian_direction

    def hook_fn(module, inputs, output):
        hidden_states = _layer_output_to_hidden_states(output)
        delta = alpha * steering_direction.to(hidden_states.dtype).view(1, 1, -1)
        steered_hidden_states = hidden_states + delta
        if isinstance(output, tuple):
            return (steered_hidden_states, *output[1:])
        return steered_hidden_states

    return hook_fn


def generate_with_italian_feature_edit(
    question: str,
    prompt_style: str,
    layer: int,
    alpha: float = 4.0,
    n_features: int = 10,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    selected_italian_features = italian_feature_ids[:n_features]
    prompt = build_prompt(question, prompt_style)
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    hook = make_italian_feature_hook(
        layer=layer,
        feature_ids=selected_italian_features,
        alpha=alpha
    )
    handle = model.model.layers[layer].register_forward_hook(hook)
    try:
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
intervention_cases = []
for question in english_test_questions[:5]:
    intervention_cases.append({
        "question_set": "english_questions",
        "prompt_style": "english",
        "question": question,
        "intervention_alpha": 4.0,
        "condition": "English prompt + positive Italian-feature alpha",
    })
for question in italian_test_questions[:5]:
    intervention_cases.append({
        "question_set": "italian_questions",
        "prompt_style": "italian",
        "question": question,
        "intervention_alpha": -4.0,
        "condition": "Italian prompt + negative Italian-feature alpha",
    })

N_ITALIAN_FEATURES = 10

intervention_rows = []
for case in tqdm(intervention_cases, desc="SAE interventions"):

    response = generate_with_italian_feature_edit(
        question=case["question"],
        prompt_style=case["prompt_style"],
        layer=PRIMARY_SAE_LAYER,
        alpha=case["intervention_alpha"],
        n_features=N_ITALIAN_FEATURES,
    )
    intervention_rows.append({
        "question_set": case["question_set"],
        "prompt_style": case["prompt_style"],
        "condition": case["condition"],
        "alpha": case["intervention_alpha"],
        "question": case["question"],
        "detected_language": classify_language_by_markers(response),
        "response": response,
    })

intervention_results = pd.DataFrame(intervention_rows)
intervention_results

In [ ]:
count_table = (
    intervention_results
    .groupby(["condition", "alpha", "detected_language"])
    .size()
    .reset_index(name="n_responses")
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (condition, subset) in zip(axes, count_table.groupby("condition", sort=False)):
    pivot = (
        subset
        .pivot(index="alpha", columns="detected_language", values="n_responses")
        .reindex(columns=LANGUAGE_ORDER, fill_value=0)
        .fillna(0)
    )
    colors = [LANGUAGE_COLORS[language] for language in pivot.columns]
    pivot.plot(kind="bar", ax=ax, color=colors, rot=0, width=0.75)
    ax.set_title(condition)
    ax.set_xlabel("Intervention strength alpha")
    ax.set_ylabel("Number of responses")
    ax.legend(title="Detected language", frameon=False)

fig.suptitle("Classified response language under Italian-feature intervention", y=1.04)
plt.tight_layout()
plt.show()

## 5. Inspect Feature Activations on Unseen Text

A large differential score does not automatically make a feature interpretable. An additional sanity check is to inspect whether the selected features activate consistently on new examples that were not used to discover them.

We encode unseen English and Italian sentences with the same layer-4 SAE, then examine which sentences produce the largest activation $z_j$ for the selected English- and Italian-associated features. This helps us check whether the hypothesized language features generalize beyond the original contrastive dataset.

In [ ]:
def encode_sentences_with_sae(
    sentences: list[str],
    layer: int = PRIMARY_SAE_LAYER,
    batch_size: int = BATCH_SIZE,
) -> torch.Tensor:
    residuals = extract_residuals_by_layer(
        sentences, layers=[layer], batch_size=batch_size
    )[layer]
    return encode_topk_sae(residuals, sae_by_layer[layer])

unseen_sae_activations = {
    "english": encode_sentences_with_sae(english_unseen_sentences),
    "italian": encode_sentences_with_sae(italian_unseen_sentences),
}
print({
    language: tuple(activations.shape)
    for language, activations in unseen_sae_activations.items()
})

In [ ]:
def top_unseen_examples(feature_id: int, top_n: int = 10) -> pd.DataFrame:
    rows = []
    for language, feature_activations, sentences in [
        ("english", unseen_sae_activations["english"][:, feature_id], english_unseen_sentences),
        ("italian", unseen_sae_activations["italian"][:, feature_id], italian_unseen_sentences),
    ]:
        values, indices = torch.topk(feature_activations, k=min(top_n, len(feature_activations)))
        for activation, sentence_index in zip(values.tolist(), indices.tolist()):
            rows.append({
                "feature_id": feature_id,
                "language": language,
                "activation": activation,
                "sentence": sentences[sentence_index],
            })
    return pd.DataFrame(rows).sort_values("activation", ascending=False).reset_index(drop=True)

english_feature_id = english_feature_ids[0]
print(f"Top unseen examples for English-associated feature L{PRIMARY_SAE_LAYER}/F{english_feature_id}")
display(top_unseen_examples(english_feature_id))

italian_feature_id = italian_feature_ids[0]
print(f"Top unseen examples for Italian-associated feature L{PRIMARY_SAE_LAYER}/F{italian_feature_id}")
display(top_unseen_examples(italian_feature_id))

## Summary

Notebook 01 represented language with one dense residual-stream direction. In this notebook, we motivated SAEs as a way to address superposition by replacing the original residual coordinates with a large, sparse dictionary of learned feature directions.

For a TopK SAE,

$$
z = \operatorname{TopK}(W_{\text{enc}}h + b_{\text{enc}}),
\qquad
\hat{h} = b_{\text{dec}} + \sum_j z_j d_j.
$$

The differential score $\Delta_j = \mathbb{E}[z_j \mid \text{Italian}] - \mathbb{E}[z_j \mid \text{English}]$ identifies features associated with Italian text. We use these TopK activations for feature discovery. The resulting fixed direction $v_{\text{it}} = \sum_{j \in I} d_j$ is added to encourage Italian and subtracted to suppress it, with $\alpha$ controlling the overall intervention strength. The unseen-text analysis then checks whether the discovered feature interpretation generalizes beyond the sentences used to find it.


## Optional Extension: Try Another Language

The main lab uses English and Italian, however, the `data/` directory also contains parallel translations of the same examples in **German, Spanish, Chinese, and Turkish**.

Each additional language provides:

- `<language>_sentences.json` for contrastive feature discovery;
- `<language>_test_questions.json` for generation experiments;
- `<language>_unseen_sentences.json` for the final feature-activation inspection.

The examples appear in the same order as their English counterparts. This makes it possible to repeat the analysis while keeping sentence content approximately controlled and changing the language.

To experiment with another language, replace the Italian files loaded at the beginning of the notebook and adapt the Italian-specific prompts, marker-word classifier, variable labels, and plot text. The mathematics does not change: you can compute `mean(chosen language) - mean(English)` using either the residual stream vectors from Notebook 01, or using the SAE hidden activations to identify SAE features for the chosen language. You can then repeat the intervention experiments to see if you can replicate the results across languages!